# DoS MuMoRAG Attack on the whole ViDoRe Benchmark

### imports

In [10]:
from datasets import load_dataset, Dataset
import torchvision.transforms as T
from transformers import AutoModelForZeroShotImageClassification, AutoModel, AutoProcessor, AutoTokenizer
from utils.utils import get_device, add_img_embedding_column, retrieve_images_by_prompt, plot_images
from utils.image_utils import process_image
from utils.model_utils import test_embeddings_loss, load_emb_model, compute_txt_embedding, compute_img_embedding
import torch
from attack import rag_attack
from collections import defaultdict

### Load Dataset

In [ ]:
ds = load_dataset("vidore/syntheticDocQA_artificial_intelligence_test", split='test')
ds

Dataset({
    features: ['query', 'image', 'image_filename', 'answer', 'page', 'model', 'prompt', 'source'],
    num_rows: 1000
})

### Load Embedding Model

In [ ]:
device = get_device(prefer_mps=True)
model_name = "openai/clip-vit-base-patch16" # "openai/clip-vit-base-patch16" or "nomic-ai/nomic-embed-vision-v1.5", "jinaai/jina-clip-v2" (large), "jinaai/jina-clip-v1" 
emb_loss_type = "mse" # mse, l2

model, processor, tokenizer = load_emb_model(model_name, device)

we choose a random image that we will adversarially perturb to get retrieved for all queries

In [13]:
chosen_index = 250
chosen_image = T.PILToTensor()(ds[chosen_index]['image']) # choose from after 100 since those do not have associated queries
chosen_image.shape

torch.Size([3, 2200, 1700])

### Computing embeddings for the queries and their correct images

In [ ]:
query_strings = ds[:100]['query']
images = ds[:100]['image']

query_embeddings = compute_txt_embedding(query_strings, model_name, model, tokenizer, processor, device)
correct_img_embeddings = compute_img_embedding(images, None, model_name, model, tokenizer, processor, device, overwrite=False)

query_embeddings.shape, correct_img_embeddings.shape

(torch.Size([100, 512]), torch.Size([100, 512]))

In [15]:
target_mses = torch.tensor([torch.nn.functional.mse_loss(qe, ie).item() for qe, ie in zip(query_embeddings, correct_img_embeddings)])
len(target_mses), target_mses

(100,
 tensor([0.2753, 0.2872, 0.2549, 0.3176, 0.2653, 0.2342, 0.2048, 0.2919, 0.2211,
         0.2683, 0.2505, 0.2823, 0.2696, 0.2880, 0.2749, 0.3200, 0.2603, 0.2714,
         0.3181, 0.2794, 0.3023, 0.3046, 0.3048, 0.2679, 0.2368, 0.2865, 0.2384,
         0.3194, 0.2761, 0.2825, 0.2077, 0.3160, 0.2620, 0.2844, 0.2784, 0.2455,
         0.2926, 0.2826, 0.2834, 0.2355, 0.2883, 0.3070, 0.3131, 0.2308, 0.3023,
         0.2802, 0.2663, 0.2236, 0.3036, 0.2685, 0.2909, 0.2964, 0.3518, 0.2476,
         0.2806, 0.1927, 0.3359, 0.2433, 0.2639, 0.2517, 0.2533, 0.2896, 0.2726,
         0.2620, 0.2636, 0.3037, 0.2356, 0.3336, 0.2427, 0.2249, 0.2756, 0.2679,
         0.2686, 0.2333, 0.2402, 0.2146, 0.3297, 0.2952, 0.2712, 0.2533, 0.2395,
         0.3273, 0.2855, 0.3043, 0.3171, 0.2576, 0.3044, 0.2919, 0.2831, 0.2801,
         0.2989, 0.2970, 0.2364, 0.2670, 0.2840, 0.2420, 0.3150, 0.2704, 0.2645,
         0.2706]))

In [16]:
initial_chosen_image = chosen_image.clone()

chosen_img_embeddings = compute_img_embedding(chosen_image, chosen_image, model_name, model, tokenizer, processor, device, overwrite=True)
initial_mses = torch.tensor([torch.nn.functional.mse_loss(qe, chosen_img_embeddings[0]).item() for qe in query_embeddings])
initial_mses

tensor([0.3003, 0.3096, 0.2948, 0.3426, 0.3112, 0.2842, 0.2895, 0.3127, 0.2765,
        0.3175, 0.3340, 0.3030, 0.3135, 0.2993, 0.3325, 0.3283, 0.3033, 0.3274,
        0.3242, 0.3076, 0.3399, 0.3404, 0.3320, 0.3135, 0.2929, 0.3176, 0.3113,
        0.3410, 0.3222, 0.3211, 0.3026, 0.3336, 0.2802, 0.3222, 0.2950, 0.2951,
        0.3189, 0.2961, 0.3616, 0.3397, 0.3281, 0.3176, 0.3184, 0.3061, 0.3161,
        0.3258, 0.3055, 0.3196, 0.3101, 0.3234, 0.3337, 0.3144, 0.3665, 0.3174,
        0.3427, 0.2920, 0.3400, 0.3025, 0.2834, 0.3130, 0.3063, 0.3251, 0.3179,
        0.2998, 0.3076, 0.3416, 0.3211, 0.3599, 0.2984, 0.2791, 0.3076, 0.2858,
        0.2924, 0.3153, 0.2789, 0.3190, 0.3121, 0.3150, 0.3103, 0.3082, 0.2917,
        0.3170, 0.3027, 0.3405, 0.3495, 0.3162, 0.3206, 0.3356, 0.3372, 0.3152,
        0.3123, 0.3344, 0.3088, 0.3043, 0.3311, 0.3059, 0.3324, 0.2953, 0.2988,
        0.3131])

### Compute the adversarial attack

In [17]:
chosen_image = chosen_image.float()
image_adv = rag_attack(
    raw_image=chosen_image,
    emb_model_name=model_name,
    model_emb=model,
    processor_emb=processor,
    tokenizer_emb=tokenizer,
    vlm_model_name=None,
    model_vlm=None,
    processor_vlm=None,
    user_query=query_strings,
    target_answer=None,
    max_perturbation=0.05,
    n_iter=100,
    lr=255*(3e-3),
    lambda_emb=1,
    lambda_vlm=0,
    emb_loss_type=emb_loss_type,
    device=device
    )

/Users/ezaki/Library/CloudStorage/OneDrive-TheAlanTuringInstitute/Research/LLMSec/code/mumorag_gh/mumoRAG-attacks/src/utils/model_utils.py:122: UserWarning: Using a target size (torch.Size([100, 512])) that is different to the input size (torch.Size([1, 512])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return torch.nn.functional.mse_loss(image_embedding, text_embedding)


Iter    1, Losses -> Embedding: 0.31429201, VLM: 0.00000000, Total: 0.31429201
Iter   10, Losses -> Embedding: 0.19995454, VLM: 0.00000000, Total: 0.19995454
Iter   20, Losses -> Embedding: 0.18853511, VLM: 0.00000000, Total: 0.18853511
Iter   30, Losses -> Embedding: 0.18038219, VLM: 0.00000000, Total: 0.18038219
Iter   40, Losses -> Embedding: 0.17614619, VLM: 0.00000000, Total: 0.17614619
Iter   50, Losses -> Embedding: 0.17288116, VLM: 0.00000000, Total: 0.17288116
Iter   60, Losses -> Embedding: 0.17222415, VLM: 0.00000000, Total: 0.17222415
Iter   70, Losses -> Embedding: 0.17142853, VLM: 0.00000000, Total: 0.17142853
Iter   80, Losses -> Embedding: 0.16761214, VLM: 0.00000000, Total: 0.16761214
Iter   90, Losses -> Embedding: 0.16904861, VLM: 0.00000000, Total: 0.16904861
Iter  100, Losses -> Embedding: 0.16794048, VLM: 0.00000000, Total: 0.16794048


In [18]:
target_mses = target_mses.to(device)

init_mses = test_embeddings_loss(initial_chosen_image, query_strings, model_name, model, processor, tokenizer, device, loss_type=emb_loss_type)
adv_mses = test_embeddings_loss(image_adv.type(torch.int32), query_strings, model_name, model, processor, tokenizer, device, loss_type=emb_loss_type)
adv_mses_ow = test_embeddings_loss(image_adv.type(torch.int32), query_strings, model_name, model, processor, tokenizer, device, overwrite=True, loss_type=emb_loss_type)

num_success_init = sum(init_mses < target_mses).item()
num_success_adv = sum(adv_mses < target_mses).item()
num_success_adv_ow = sum(adv_mses_ow < target_mses).item()

num_success_init, num_success_adv, num_success_adv_ow

(11, 100, 100)

### Update dataset with the adversarial image and prepare for RAG

In [19]:
ds_dict = defaultdict(list)
for i in range(len(ds)):
    ds_dict['image'].append(ds[i]['image'])
    ds_dict['index'].append(i)
ds_dict['image'].append(T.ToPILImage()(image_adv/255))
ds_dict['index'].append(len(ds))
ds_adv = Dataset.from_dict(ds_dict)

ds_adv_with_embeddings = add_img_embedding_column(ds_adv, model, processor, existing_col_name="image", new_col_name="image_embeddings", device=device)
ds_adv_with_embeddings.add_faiss_index(column="image_embeddings")

Map:   0%|          | 0/1001 [00:00<?, ? examples/s]

  0%|          | 0/2 [00:00<?, ?it/s]

Dataset({
    features: ['image', 'index', 'image_embeddings'],
    num_rows: 1001
})

### Retrieve most relevant image for each query

In [20]:
# plot_images([ds_adv_with_embeddings[250]['image'], ds_adv_with_embeddings[1000]['image'], T.ToPILImage()(image_adv/255)], n_subplots=3)
# image_adv

In [21]:
topk=1

indices = []
for i in range(100):
    scores, retrieved = retrieve_images_by_prompt(query_strings[i], ds_adv_with_embeddings, model, tokenizer, topk, device, plot=False)
    indices.append(retrieved["index"][0])

print(f"Number of queries for which the adversarial image is retrieved: {sum(torch.tensor(indices) == 1000).item()}/100")

Number of queries for which the adversarial image is retrieved: 100/100
